# Training — Zou et al. 2017 Replication
Runs all three experiments from the paper:
1. Single-modal fALFF
2. Single-modal GM
3. Multi-modal fALFF + GM (paper's best: 69.15%)

Full run: 50 repeats × 4 folds × 100 epochs. Edit `N_REPEATS` and `EPOCHS` below for a quick smoke-test.

In [6]:
import subprocess, sys, os, json, re
from pathlib import Path

SCRIPTS_DIR = Path('../scripts')
OUTPUTS_DIR = Path('../outputs')
OUTPUTS_DIR.mkdir(exist_ok=True)

# --- Experiment settings ---
N_REPEATS  = 50    # paper uses 50; set to 2 for a quick smoke-test
N_FOLDS    = 4
EPOCHS     = 100   # paper uses 100; set to 5 for a quick smoke-test
BATCH_SIZE = 8

print(f'N_REPEATS={N_REPEATS}  EPOCHS={EPOCHS}  BATCH_SIZE={BATCH_SIZE}')

N_REPEATS=50  EPOCHS=100  BATCH_SIZE=8


In [7]:
def run_experiment(label, extra_args):
    """Run train.py with given args, stream output, return parsed summary."""
    cmd = [
        sys.executable, str(SCRIPTS_DIR / 'train.py'),
        '--n-repeats',  str(N_REPEATS),
        '--n-folds',    str(N_FOLDS),
        '--epochs',     str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--output-dir', str(OUTPUTS_DIR / label.replace(' ', '_')),
    ] + extra_args

    print(f'\n{"="*60}')
    print(f'EXPERIMENT: {label}')
    print(f'CMD: {" ".join(cmd)}')
    print('='*60)

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=str(SCRIPTS_DIR),
    )
    lines = []
    for line in proc.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    proc.wait()

    # Parse summary from output
    summary = {'label': label, 'returncode': proc.returncode}
    for line in lines:
        m = re.search(r'Mean val accuracy\s*:\s*([\d.]+)%', line)
        if m: summary['mean_val_acc'] = float(m.group(1))
        m = re.search(r'Variance\s*:\s*([\d.]+)', line)
        if m: summary['variance'] = float(m.group(1))
        m = re.search(r'Best single run\s*:\s*([\d.]+)%', line)
        if m: summary['best_run'] = float(m.group(1))
    return summary

results = []

## Experiment 1 — Single-modal fALFF

In [8]:
r = run_experiment('falff', ['--mode', 'single', '--derivative', 'falff'])
results.append(r)
print(f"\nResult: {r.get('mean_val_acc', 'N/A')}% (variance {r.get('variance', 'N/A')})")


EXPERIMENT: falff
CMD: /opt/anaconda3/bin/python ../scripts/train.py --n-repeats 50 --n-folds 4 --epochs 100 --batch-size 8 --output-dir ../outputs/falff --mode single --derivative falff
Device: mps
Dataset: 63 subjects  |  mode=single  |  derivative=falff
  repeat= 1/50  fold=1/4  epoch=  1/100  tr_loss=1.4339  tr_acc=0.275  val_loss=0.7399  val_acc=0.250  [2s]
  repeat= 1/50  fold=1/4  epoch= 10/100  tr_loss=0.4986  tr_acc=0.750  val_loss=0.6045  val_acc=0.562  [6s]
  repeat= 1/50  fold=1/4  epoch= 20/100  tr_loss=0.4828  tr_acc=0.775  val_loss=0.5601  val_acc=0.750  [11s]
  repeat= 1/50  fold=1/4  epoch= 30/100  tr_loss=0.3442  tr_acc=0.825  val_loss=0.6568  val_acc=0.688  [16s]
  repeat= 1/50  fold=1/4  epoch= 40/100  tr_loss=0.4638  tr_acc=0.825  val_loss=0.6319  val_acc=0.562  [21s]
  repeat= 1/50  fold=1/4  epoch= 50/100  tr_loss=0.2725  tr_acc=0.875  val_loss=0.7159  val_acc=0.562  [26s]
  repeat= 1/50  fold=1/4  epoch= 60/100  tr_loss=0.2734  tr_acc=0.850  val_loss=0.5694  va

KeyboardInterrupt: 

## Experiment 2 — Single-modal GM

In [4]:
r = run_experiment('gm', ['--mode', 'single', '--derivative', 'gm'])
results.append(r)
print(f"\nResult: {r.get('mean_val_acc', 'N/A')}% (variance {r.get('variance', 'N/A')})")


EXPERIMENT: gm
CMD: /opt/anaconda3/bin/python ../scripts/train.py --n-repeats 50 --n-folds 4 --epochs 100 --batch-size 8 --output-dir ../outputs/gm --mode single --derivative gm
Device: mps
Dataset: 63 subjects  |  mode=single  |  derivative=gm
  repeat= 1/50  fold=1/4  epoch=  1/100  tr_loss=1.0267  tr_acc=0.475  val_loss=0.7110  val_acc=0.375  [7s]
  repeat= 1/50  fold=1/4  epoch= 10/100  tr_loss=0.3375  tr_acc=0.850  val_loss=0.7318  val_acc=0.500  [41s]
  repeat= 1/50  fold=1/4  epoch= 20/100  tr_loss=0.4166  tr_acc=0.800  val_loss=0.6537  val_acc=0.625  [77s]
  repeat= 1/50  fold=1/4  epoch= 30/100  tr_loss=0.2130  tr_acc=0.925  val_loss=0.6425  val_acc=0.750  [114s]
  repeat= 1/50  fold=1/4  epoch= 40/100  tr_loss=0.0946  tr_acc=0.975  val_loss=0.6824  val_acc=0.688  [151s]
  repeat= 1/50  fold=1/4  epoch= 50/100  tr_loss=0.1474  tr_acc=0.925  val_loss=0.7151  val_acc=0.688  [188s]
  repeat= 1/50  fold=1/4  epoch= 60/100  tr_loss=0.1105  tr_acc=0.975  val_loss=0.7139  val_acc=0.

KeyboardInterrupt: 

## Experiment 3 — Multi-modal fALFF + GM

In [ ]:
r = run_experiment('falff_gm_multi', [
    '--mode', 'multi',
    '--fmri-derivative', 'falff',
    '--smri-derivative', 'gm',
])
results.append(r)
print(f"\nResult: {r.get('mean_val_acc', 'N/A')}% (variance {r.get('variance', 'N/A')})")

## Summary

In [ ]:
import pandas as pd

# Paper targets (Table III)
paper = {
    'falff':         {'mean_val_acc': 62.06, 'variance': None},
    'gm':            {'mean_val_acc': 65.43, 'variance': None},
    'falff_gm_multi':{'mean_val_acc': 69.15, 'variance': None},
}

rows = []
for r in results:
    key = r['label']
    ours = r.get('mean_val_acc', float('nan'))
    paper_acc = paper.get(key, {}).get('mean_val_acc', float('nan'))
    rows.append({
        'Experiment':  key,
        'Our acc (%)': f"{ours:.2f}",
        'Paper (%)':   f"{paper_acc:.2f}",
        'Δ':           f"{ours - paper_acc:+.2f}",
        'Variance':    f"{r.get('variance', float('nan')):.2f}",
        'Best run (%)':f"{r.get('best_run', float('nan')):.2f}",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Save summary
summary_path = OUTPUTS_DIR / 'summary.json'
with open(summary_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved to {summary_path}')

## Accuracy over repeats (per experiment)

In [5]:
import matplotlib.pyplot as plt

# Read per-repeat means from each results.txt
exp_data = {}
for r in results:
    results_file = OUTPUTS_DIR / r['label'].replace(' ', '_') / 'results.txt'
    if results_file.exists():
        with open(results_file) as f:
            for line in f:
                if line.startswith('all_repeat_means:'):
                    import ast
                    vals = ast.literal_eval(line.split(':', 1)[1].strip())
                    exp_data[r['label']] = vals

fig, ax = plt.subplots(figsize=(10, 4))
for label, vals in exp_data.items():
    ax.plot(range(1, len(vals)+1), vals, marker='.', label=label)

# Paper baselines
paper_lines = {'falff': 62.06, 'gm': 65.43, 'falff_gm_multi': 69.15}
colors = ['C0', 'C1', 'C2']
for (k, v), c in zip(paper_lines.items(), colors):
    ax.axhline(v, linestyle='--', color=c, alpha=0.5, label=f'{k} paper target')

ax.set_xlabel('Repeat')
ax.set_ylabel('Val accuracy (%)')
ax.set_title('Validation accuracy per repeat')
ax.legend(loc='lower right', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

ValueError: malformed node or string on line 1: <ast.Call object at 0x12147ff50>